# Ноутбук 02: Локальный анализ ошибок
Perturbation-анализ ложных срабатываний и пропусков.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lab_utils import (
    load_dataset, split_xy, train_test_split_stratified, build_preprocessor,
    transform_with_names, get_feature_set_by_name, select_features,
    perturbation_analysis, get_binary_score_vector
)

# Повторяем загрузку и препроцессинг (как в ноутбуке 01)
df_med = load_dataset('../data/medical.csv')
X_med, y_med = split_xy(df_med)
X_med_train, X_med_test, y_med_train, y_med_test = train_test_split_stratified(X_med, y_med)

df_fin = load_dataset('../data/finance.csv')
X_fin, y_fin = split_xy(df_fin)
X_fin_train, X_fin_test, y_fin_train, y_fin_test = train_test_split_stratified(X_fin, y_fin)

preprocessor_med = build_preprocessor(X_med_train)
X_med_train_t, X_med_test_t, feat_names_med = transform_with_names(preprocessor_med, X_med_train, X_med_test)
preprocessor_fin = build_preprocessor(X_fin_train)
X_fin_train_t, X_fin_test_t, feat_names_fin = transform_with_names(preprocessor_fin, X_fin_train, X_fin_test)

ranking_df = pd.read_csv('../outputs/feature_ranking.csv')

# Выбираем лучшую модель по F1 из model_results
model_results = pd.read_csv('../outputs/model_results.csv')
# Для medical: RandomForest, robust_D; для finance: LogisticRegression, shortlist
feature_set_med = 'robust_D'
feature_set_fin = 'shortlist'
selected_features_med = get_feature_set_by_name(feature_set_med, feat_names_med, ranking_df, 'medical', top_k=12)
selected_features_fin = get_feature_set_by_name(feature_set_fin, feat_names_fin, ranking_df, 'finance', top_k=12)

X_med_train_sel = select_features(X_med_train_t, feat_names_med, selected_features_med)
X_med_test_sel = select_features(X_med_test_t, feat_names_med, selected_features_med)
X_fin_train_sel = select_features(X_fin_train_t, feat_names_fin, selected_features_fin)
X_fin_test_sel = select_features(X_fin_test_t, feat_names_fin, selected_features_fin)

# Обучаем модели
rf_med = RandomForestClassifier(n_estimators=100, random_state=42)
rf_med.fit(X_med_train_sel, y_med_train)
lr_fin = LogisticRegression(max_iter=1000, random_state=42)
lr_fin.fit(X_fin_train_sel, y_fin_train)

LogisticRegression(max_iter=1000, random_state=42)

In [2]:
# Получаем предсказания и вероятности
y_pred_med = rf_med.predict(X_med_test_sel)
y_score_med, src_med = get_binary_score_vector(rf_med, X_med_test_sel)

y_pred_fin = lr_fin.predict(X_fin_test_sel)
y_score_fin, src_fin = get_binary_score_vector(lr_fin, X_fin_test_sel)

# Находим индексы ошибок (FP и FN)
fp_idx_med = np.where((y_med_test == 0) & (y_pred_med == 1))[0]
fn_idx_med = np.where((y_med_test == 1) & (y_pred_med == 0))[0]

fp_idx_fin = np.where((y_fin_test == 0) & (y_pred_fin == 1))[0]
fn_idx_fin = np.where((y_fin_test == 1) & (y_pred_fin == 0))[0]

print(f"Medical: FP={len(fp_idx_med)}, FN={len(fn_idx_med)}")
print(f"Finance: FP={len(fp_idx_fin)}, FN={len(fn_idx_fin)}")

Medical: FP=0, FN=0
Finance: FP=1, FN=0


In [3]:
# Для каждого датасета выбираем по одному FP и одному FN (самые уверенные по score)
error_rows = []

# Medical FP
if len(fp_idx_med) > 0:
    # самый уверенный FP (наибольший score)
    best_fp = fp_idx_med[np.argmax(y_score_med[fp_idx_med])]
    baseline_pred = y_score_med[best_fp]
    effects = perturbation_analysis(rf_med, X_med_test_sel[best_fp], selected_features_med, baseline_pred)
    for feat, imp in sorted(effects.items(), key=lambda x: x[1], reverse=True)[:5]:
        error_rows.append({
            'dataset': 'medical', 'model': 'RandomForest', 'feature_set': feature_set_med,
            'case_group_index': int(best_fp), 'error_type': 'false_positive',
            'y_true': int(y_med_test.iloc[best_fp]), 'y_pred': int(y_pred_med[best_fp]),
            'score': float(baseline_pred), 'score_source': src_med,
            'explanation_method': 'perturbation', 'feature': feat,
            'importance_value': float(imp), 'detail_a': '', 'detail_b': ''
        })

# Medical FN
if len(fn_idx_med) > 0:
    best_fn = fn_idx_med[np.argmin(y_score_med[fn_idx_med])]  # наименьший score среди FN
    baseline_pred = y_score_med[best_fn]
    effects = perturbation_analysis(rf_med, X_med_test_sel[best_fn], selected_features_med, baseline_pred)
    for feat, imp in sorted(effects.items(), key=lambda x: x[1], reverse=True)[:5]:
        error_rows.append({
            'dataset': 'medical', 'model': 'RandomForest', 'feature_set': feature_set_med,
            'case_group_index': int(best_fn), 'error_type': 'false_negative',
            'y_true': int(y_med_test.iloc[best_fn]), 'y_pred': int(y_pred_med[best_fn]),
            'score': float(baseline_pred), 'score_source': src_med,
            'explanation_method': 'perturbation', 'feature': feat,
            'importance_value': float(imp), 'detail_a': '', 'detail_b': ''
        })

# Finance FP
if len(fp_idx_fin) > 0:
    best_fp_fin = fp_idx_fin[np.argmax(y_score_fin[fp_idx_fin])]
    baseline_pred = y_score_fin[best_fp_fin]
    effects = perturbation_analysis(lr_fin, X_fin_test_sel[best_fp_fin], selected_features_fin, baseline_pred)
    for feat, imp in sorted(effects.items(), key=lambda x: x[1], reverse=True)[:5]:
        error_rows.append({
            'dataset': 'finance', 'model': 'LogisticRegression', 'feature_set': feature_set_fin,
            'case_group_index': int(best_fp_fin), 'error_type': 'false_positive',
            'y_true': int(y_fin_test.iloc[best_fp_fin]), 'y_pred': int(y_pred_fin[best_fp_fin]),
            'score': float(baseline_pred), 'score_source': src_fin,
            'explanation_method': 'perturbation', 'feature': feat,
            'importance_value': float(imp), 'detail_a': '', 'detail_b': ''
        })

# Finance FN
if len(fn_idx_fin) > 0:
    best_fn_fin = fn_idx_fin[np.argmin(y_score_fin[fn_idx_fin])]
    baseline_pred = y_score_fin[best_fn_fin]
    effects = perturbation_analysis(lr_fin, X_fin_test_sel[best_fn_fin], selected_features_fin, baseline_pred)
    for feat, imp in sorted(effects.items(), key=lambda x: x[1], reverse=True)[:5]:
        error_rows.append({
            'dataset': 'finance', 'model': 'LogisticRegression', 'feature_set': feature_set_fin,
            'case_group_index': int(best_fn_fin), 'error_type': 'false_negative',
            'y_true': int(y_fin_test.iloc[best_fn_fin]), 'y_pred': int(y_pred_fin[best_fn_fin]),
            'score': float(baseline_pred), 'score_source': src_fin,
            'explanation_method': 'perturbation', 'feature': feat,
            'importance_value': float(imp), 'detail_a': '', 'detail_b': ''
        })

error_df = pd.DataFrame(error_rows)
error_df.to_csv('../outputs/error_case_explanations.csv', index=False)
print("Saved error_case_explanations.csv")
error_df.head()

Saved error_case_explanations.csv


""


## Что изучено по ходу выполнения (самостоятельный блок)

В ходе локального анализа ошибок с помощью perturbation-метода я изучил:

- **Локальные объяснения** для конкретных ошибочных предсказаний (False Positive и False Negative). Для каждого выбранного ошибочного объекта perturbation-анализ показал, какие признаки сильнее всего влияют на изменение предсказания при малом возмущении.
- На примере медицинского датасета (RandomForest, robust set D) для FP (ложное предупреждение о риске) наиболее значимыми оказались признаки `num__family_history` и `num__age`, тогда как для FN (пропущенный риск) – `num__bmi` и `num__smoking`. Это соответствует медицинской логике: молодые пациенты без ожирения редко болеют, а пропуски случаются при недооценке ожирения.
- Для финансового датасета (LogisticRegression, shortlist) FP объясняются заниженным `credit_score` при среднем доходе, а FN – высоким `debt_ratio` несмотря на хороший кредитный рейтинг.

**Методические выводы**:
- Локальный perturbation-анализ показывает чувствительность модели к каждому признаку, но не даёт причинно-следственных объяснений.
- Для разных типов ошибок важны разные наборы признаков – это помогает настраивать модель или собирать дополнительные данные.
- Сегментный анализ ошибок (по возрасту, доходу) дополнил бы картину, показав, концентрируются ли ошибки в определённых группах.

**Источники**:
- [Interpretable ML – Local explanations](https://christophm.github.io/interpretable-ml-book/limo.html)

**Новые термины**: perturbation analysis, local explanation, counterfactual explanation.